# UNETR LR 3e-4 baseline — full-budget continuation
Resume the exact fold 0 / seed 0 checkpoint at update **207**, through **414 total updates**.
Only the halfway gate is removed. Same ViT-B/16, Dice+CE 1:1, XY D4, initialization, optimizer, precision, data and original cosine schedule.
**182,178.12654316 GFLOP cap.** No fresh start or additional training arms.
Missing recovery stops execution. Prior pruning history remains; use `lr3_full_budget_status.json`.
Account focacciafilippo@gmail.com; same A100 80GB environment.


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()
from googleapiclient.discovery import build
import google.auth
credentials, _ = google.auth.default()
service = build('drive','v3',credentials=credentials,cache_discovery=False)
about = service.about().get(fields='user(emailAddress),storageQuota').execute()
assert about['user']['emailAddress'].lower() == 'focacciafilippo@gmail.com', 'Mount/authenticate the requested account.'
quota=about['storageQuota']
if 'limit' in quota:
    assert int(quota['limit'])-int(quota['usage']) >= 8*1024**3, 'Keep at least 8 GiB free in Drive, including trash.'
from pathlib import Path
import hashlib,json,os,shutil,signal,subprocess,sys,time,zipfile
HOME_DIR=Path('/content/drive/MyDrive/hippo_geofwi3d_unetr_ce_pilot_20261019')
INPUT=HOME_DIR/'input'
RESULTS=HOME_DIR/'results_unetr_lr3_bands_f0s0_v1'
BUNDLE=Path('/content/hippo_geofwi_unetr_lr3_bands_v1')
RESULTS.mkdir(parents=True,exist_ok=True)
BUNDLE.mkdir(exist_ok=True)
assert shutil.disk_usage('/content').free >= 24*1024**3, 'Need at least 24 GiB runtime disk space.'
print('Account:',about['user']['emailAddress'],'Results:',RESULTS)


## Verify and stage the uploaded data/code
All archive and per-file SHA256 hashes are checked. The 319 unique images/labels are read from
the runtime SSD during training. Existing matching files are reused; changed files stop execution.


In [ ]:
EXPECTED_UPLOADS={'geofwi3d_data_part01.zip': '32eaadf1a6abd61dd5bd922a8a085af4e6c876b34b915bbd09da93a3dbea2ca2', 'geofwi3d_data_part02.zip': '31adc6fa680b74a921f9457f24d2dfbb178aa1cef1d80baa152c5ee82a4e05bd', 'geofwi3d_data_part03.zip': 'e9c404631c47a918d1ca36c9166ea92141aa1b8264e332fbaa32b13299d8706a', 'geofwi3d_data_part04.zip': '1ef1e965ebefb571f441d986f2304edb4f2ab824b4bedb7dfbc6a494c4c1c279', 'geofwi3d_data_part05.zip': 'b99cbe88932b94bc382512c059661b48516beacd1dae116237ec411edc64fb58', 'geofwi3d_data_part06.zip': '6f0509326647688300459b05d5f0af5c10ecdde9c19fc6fc3ee05bc92fe8f183', 'geofwi3d_data_part07.zip': '3e1cf8a1cacd445c73bf1798ebd9c1b3c34f6f2cea0e2bf0ad378c6f12f09fb1', 'geofwi3d_data_part08.zip': 'b2b603f5ef55eb2822e64904ce7d178b7cd459c1258398de1f75cdedad5525ed', 'geofwi3d_data_part09.zip': '833fccd13996d6d6ddd118b6146334a902e4eceba2dedd26b798db64fcf6de38', 'geofwi3d_data_part10.zip': '99daf1d8f684ac47897aaa6933b69741e826935783bc42dbd02093efc299e235', 'geofwi3d_data_part11.zip': 'ce12340263414416fcd728e7fad43a2dbbbf4d26f22cc51273acb8df50680239', 'geofwi3d_data_part12.zip': 'ff0f922b1ae82afb2394725bcd0374b4eb273fe0333f5fbad4dbd52b540aadd2', 'geofwi3d_data_part13.zip': '9ede65d6da572c0186b51b838f46f7cd281bbd8b515aef813e502b59891cc880', 'geofwi3d_unetr_lr3_bands_code_v1.zip': '353014571fd3350ef7abd2c65fa8d6171f72647c15ccfd196b220dda9f886279'}
EXPECTED_MANIFEST='92dcadf14974b1b50b60573d7900c979ee59729a325f44b15c63a35af9ed9d14'
existing=Path('/content/hippo_geofwi_unetr_xy_aug_v1/data')
if existing.is_dir() and not (BUNDLE/'data').exists():
    (BUNDLE/'data').symlink_to(existing,target_is_directory=True)
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024**2),b''):h.update(chunk)
    return h.hexdigest()
for name,expected in EXPECTED_UPLOADS.items():
    package=INPUT/name
    if not package.exists():raise FileNotFoundError(str(package))
    assert sha(package)==expected, 'Upload checksum mismatch: '+name
    local=Path('/content')/name
    if not local.exists() or sha(local)!=expected:
        shutil.copyfile(package,local)
    assert sha(local)==expected
    with zipfile.ZipFile(local) as z:
        assert len(z.namelist())==len(set(z.namelist())), 'Duplicate archive members'
        for info in z.infolist():
            p=Path(info.filename)
            assert not p.is_absolute() and '..' not in p.parts
            assert (info.external_attr>>16)&0o170000 != 0o120000
        for info in z.infolist():
            target=BUNDLE/info.filename
            if target.exists():continue
            target.parent.mkdir(parents=True,exist_ok=True)
            temp=target.with_name(target.name+'.extracting')
            with z.open(info) as src,temp.open('wb') as dst:shutil.copyfileobj(src,dst)
            temp.replace(target)
    print('Staged',name)
assert sha(BUNDLE/'bundle_manifest.json')==EXPECTED_MANIFEST
for listing in ['bundle_manifest.json','data_manifest.json']:
    for name,expected in json.loads((BUNDLE/listing).read_text()).items():
        assert sha(BUNDLE/name)==expected, 'Extracted file mismatch: '+name
source=RESULTS/'provenance'/EXPECTED_MANIFEST;source.mkdir(parents=True,exist_ok=True)
for name in ['geofwi3d_unetr_lr3_bands_code_v1.zip']:
    dst=source/name
    if not dst.exists():shutil.copyfile(INPUT/name,dst)
    assert sha(dst)==EXPECTED_UPLOADS[name]
print('All inputs verified. Training augmentation: XY D4 only.')


## Install the frozen environment
Uses the package versions from the existing Colab training environment, in a separate venv.
Setup output is streamed and saved in Drive. Keep the same GPU type/software on resume.


In [ ]:
VENV=Path('/content/hippo_geofwi_unetr_ce_pilot_venv');PYTHON=VENV/'bin/python'
def command(args,log_name):
    with (RESULTS/log_name).open('a') as log:
        proc=subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,bufsize=1,start_new_session=True)
        tail=[]
        def drain():
            for line in proc.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
                tail.append(line);del tail[:-40]
        try:drain();code=proc.wait()
        except KeyboardInterrupt:
            # Signal the orchestrator only; it stops the active trainer at a safe boundary.
            proc.send_signal(signal.SIGINT)
            print('Saving recovery to Drive. Please wait; do not interrupt again.')
            drain();code=proc.wait()
        finally:proc.stdout.close()
        if code:raise RuntimeError('Process failed; recovery retained.\n'+''.join(tail))
if not PYTHON.exists():
    command([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV],'setup.log')
if subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True).returncode:
    command([PYTHON,'-m','ensurepip','--upgrade'],'setup.log')
command([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',BUNDLE/'requirements-colab.txt'],'setup.log')
def campaign(action,*extra):
    command([PYTHON,'-u',BUNDLE/'lr3_runtime.py',action,'--bundle',BUNDLE,'--durable',RESULTS,*extra],action+'.log')
campaign('verify')
subprocess.run(['nvidia-smi'],check=True)


## Restore the verified halfway checkpoint and prepare continuation
Search only the exact recovery archive, including Drive trash; validate its hash before use. Existing later recovery takes precedence.


In [ ]:
import importlib.util
from googleapiclient.http import MediaIoBaseDownload
sys.path.insert(0,str(BUNDLE))
import storage_runtime as recovery_io
run_id='GeoFWI3D_unetr_lr3_f0s0'
recovery=RESULTS/'recovery'/run_id
screen=RESULTS/'screening'/run_id
receipt=json.loads((screen/'screen_manifest.json').read_text())['recovery']
assert receipt['charged_attempts']==207
assert receipt['sha256']=='b6812e7557092c0ede742c1905df7369f140fb4e28c4efb6f1a5a3bcb9cf7c65'
output=BUNDLE/'work'/run_id
completed=RESULTS/'runs'/run_id/'cleanup.json'
if not completed.exists() and not (output/'checkpoint_latest.pt').exists() and not (recovery/'latest.json').exists():
    recovery.mkdir(parents=True,exist_ok=True)
    archive=recovery/receipt['file']
    if not archive.exists():
        # files.list includes trashed items when no trashed=false predicate is supplied.
        found=service.files().list(q="name = '"+receipt['file']+"'",fields='files(id,name,size,trashed,parents)',pageSize=100).execute()['files']
        found=[f for f in found if '1uvA4fpxvVNKuG87oF1njgwSAE847-iDW' in f.get('parents',[])]
        assert len(found)==1, 'Exact halfway archive not found uniquely; stop without retraining.'
        print('Recovering exact saved archive; trashed:',found[0].get('trashed'))
        local=Path('/content')/receipt['file']
        with local.open('wb') as stream:
            downloader=MediaIoBaseDownload(stream,service.files().get_media(fileId=found[0]['id']),chunksize=32*1024**2)
            done=False
            while not done:
                _,done=downloader.next_chunk()
        assert sha(local)==receipt['sha256'], 'Recovery archive SHA mismatch'
        recovery_io.copy_verified(local,archive)
    assert sha(archive)==receipt['sha256']
    recovery_io.write(recovery/'latest.json',receipt)
    print('Verified original checkpoint at update 207 restored. No repeated training.')

FULL_DRIVER=Path('/content/geofwi_lr3_full_runtime.py')
FULL_DRIVER.write_text('"""User-authorized continuation of the existing LR 3e-4 baseline; no halfway gate."""\nimport argparse\nimport fcntl\nimport json\nfrom pathlib import Path\nimport sys\nimport time\n\n\ndef run(root, durable, hours):\n    sys.path.insert(0,str(root))\n    import lr3_runtime as campaign\n    io=campaign.io\n    root=io.activate(root)\n    campaign.verify(root)\n    campaign.preflight(root,durable,\'lr3\')\n    arm=\'lr3\';task=campaign.task(arm)\n    work=root/\'work\';work.mkdir(exist_ok=True)\n    with (work/\'augmentation.lock\').open(\'a\') as lock:\n        fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)\n        io.write(durable/\'lr3_full_budget_authorization.json\',dict(\n            instruction=\'resume the 3e-4 baseline from halfway to full budget; remove only the halfway gate\',arm=arm,halfway_gate=False,\n            cap_flops=campaign.CAP,total_updates=414,recipe_changed=False,\n            driver_sha256=io.digest(__file__),time=time.time()))\n        result=campaign.final_result(root,durable,arm)\n        if result is None:\n            out=work/task[\'id\'];recovery=durable/\'recovery\'/task[\'id\']\n            if not (out/\'checkpoint_latest.pt\').exists():io.restore_recovery(out,recovery)\n            if not (out/\'checkpoint_latest.pt\').exists():\n                raise RuntimeError(\'No verified checkpoint available. Refusing to repeat the first 207 updates.\')\n            campaign.check_recipe(out,arm)\n            ledger=io.read(out/\'budget_ledger.json\')\n            if ledger[\'charged_attempts\']<207:raise ValueError(\'Recovery predates the halfway checkpoint\')\n            full=campaign.settings(root,arm)[2]\n            if full!=414:raise ValueError(\'Full-budget schedule changed\')\n            io.write(durable/\'lr3_full_budget_status.json\',dict(status=\'running\',\n                starting_updates=ledger[\'charged_attempts\'],target_updates=full,halfway_gate=False,\n                cap_flops=campaign.CAP,recipe_changed=False))\n            print(\'RESUME LR3\',ledger[\'charged_attempts\'],\'->\',full,\'updates; no halfway gate\',flush=True)\n            if not campaign.execute(root,durable,arm,full,time.time()+hours*3600):\n                io.write(durable/\'lr3_full_budget_status.json\',dict(status=\'paused\',target_updates=full))\n                return\n            result=campaign.final_result(root,durable,arm)\n        if not result:raise RuntimeError(\'Full-budget archive not verified\')\n        baseline=campaign.reference_result(root)\n        delta=100*(result[\'final_dice\']-baseline[\'final_dice\'])\n        value=dict(status=\'complete\',reference_lr1_baseline=baseline,lr3_baseline=result,\n            final_dice_delta_percentage_points=delta,threshold_passed=campaign.qualifies(result),\n            cap_flops=campaign.CAP,halfway_gate=False,\n            note=\'Full-budget LR3 continuation; prior pruning history retained separately; no additional training arms\')\n        io.write(durable/\'lr3_full_budget_status.json\',value)\n        io.write(durable/\'LR3_FULL_BUDGET_FINISHED.json\',value)\n        print(json.dumps(value,indent=2),flush=True)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--bundle\',type=Path,required=True)\n    parser.add_argument(\'--durable\',type=Path,required=True)\n    parser.add_argument(\'--hours\',type=float,default=2)\n    args=parser.parse_args()\n    if not 0<args.hours<=2:parser.error(\'Session limit must be in (0,2] hours\')\n    run(args.bundle.resolve(),args.durable.resolve(),args.hours)\n')
assert sha(FULL_DRIVER)=='04e41c5a049790a529780c1da6cb37f4391abccc8d9d6c2c3642f9cd05a3b101'
recovery_io.copy_verified(FULL_DRIVER,RESULTS/'provenance'/'lr3_full_runtime.py')


## Resume LR3 directly to full budget
No halfway gate. Restore the existing optimizer, RNG state and charge ledger; finish at 414 total updates.


In [ ]:
command([PYTHON,'-u',FULL_DRIVER,'--bundle',BUNDLE,'--durable',RESULTS,'--hours','2'],'lr3_full_budget.log')


## Full-budget result
Compare final-budget Dice with the archived LR 1e-4 baseline (44.82%). Prior gated status files are history.


In [ ]:
print((RESULTS/'lr3_full_budget_status.json').read_text())
